## Step 0- Libraries Import

In [1]:
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

d:\14_GenAI\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## TESTING

In [2]:
llm = ChatOllama(model="llama3")

result = llm.invoke("HEY")
print(result.content)

Hey! How's it going?


## Step 1: INDEXING 

## Step 1(a): Documents Ingestion

In [3]:
video_id = "80M9kzbY8Ms"

try:
    api = YouTubeTranscriptApi()
    scriptsList = api.fetch(video_id=video_id, languages=['hi'])
    
except TranscriptsDisabled:
    print("No caption available for this video.")


In [4]:
# Investigating

print(scriptsList)
print("\n")
print(scriptsList.snippets)
print("\n")
print(scriptsList.snippets[0])
print("\n")
print(scriptsList.snippets[0].text)



FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='यह एक अद्भुत रात थी।', start=1.36, duration=5.36), FetchedTranscriptSnippet(text='ऐसी रात जो सिर्फ जवानी में ही मुमकिन', start=4.319, duration=5.601), FetchedTranscriptSnippet(text='होती है। प्रिय पाठक', start=6.72, duration=7.12), FetchedTranscriptSnippet(text='आसमान सितारों से भरा हुआ था। इतना चमकदार', start=9.92, duration=7.84), FetchedTranscriptSnippet(text='उसे देखकर कोई यह सोचने पर मजबूर हो जाए', start=13.84, duration=6.96), FetchedTranscriptSnippet(text='कि क्या कोई चिड़चिड़ा या सनके इंसान ऐसे', start=17.76, duration=6.8), FetchedTranscriptSnippet(text='आसमान के नीचे रह सकता है?', start=20.8, duration=8.479), FetchedTranscriptSnippet(text='यह भी एक युवा सोच है। प्रिय पाठक बहुत', start=24.56, duration=7.2), FetchedTranscriptSnippet(text='युवा सोच।', start=29.279, duration=5.041), FetchedTranscriptSnippet(text='लेकिन ईश्वर करे कि यह सवाल आपके दिल में', start=31.76, duration=6.36), FetchedTranscriptSnippet(text='भी बार-बार 

In [5]:
# Gathering only 'text'

transcipts = []

for chunk in scriptsList.snippets:
    transcipts.append(chunk.text)
print(transcipts)
print("\n")

plain_transcipts = " ".join(transcipts)
print(plain_transcipts)


# Alternatives 
print("\n")

easy = " ".join(s.text for s in scriptsList.snippets)
print(easy)

['यह एक अद्भुत रात थी।', 'ऐसी रात जो सिर्फ जवानी में ही मुमकिन', 'होती है। प्रिय पाठक', 'आसमान सितारों से भरा हुआ था। इतना चमकदार', 'उसे देखकर कोई यह सोचने पर मजबूर हो जाए', 'कि क्या कोई चिड़चिड़ा या सनके इंसान ऐसे', 'आसमान के नीचे रह सकता है?', 'यह भी एक युवा सोच है। प्रिय पाठक बहुत', 'युवा सोच।', 'लेकिन ईश्वर करे कि यह सवाल आपके दिल में', 'भी बार-बार उठे।', 'जब मैं सनकी और चिड़चिड़े लोगों की बात', 'करता हूं तो मैं अपने ही मन की हालत को', 'याद करने से खुद को रोक नहीं सकता।', 'पूरा दिन मेरा मन अजीब सी उदासी से भरा', 'रहा।', 'सुबह से ही ऐसा लगा मानो मैं अकेला हूं', 'जैसे सब कुछ सब कोई मुझे छोड़कर चले जा', 'रहे हो।', 'अब कोई मुझसे पूछ सकता है कि सब कौन?', 'मैं आठ साल से पीटर्सबर्ग में रह रहा था।', 'लेकिन मेरी कोई खास जान पहचान नहीं थी।', 'लेकिन मुझे लोगों की जान पहचान की जरूरत', 'भी क्या थी?', 'मैं पीटर्सबर्ग को वैसे ही जानता था जैसे', 'कोई किसी दोस्त को जानता है। यही वजह है', 'कि जब गर्मी के दिनों में पूरा शहर अपना', 'सामान समेट कर गांव चला जाता तो मुझे लगता', 'कि सब मुझे अकेला छोड़कर च

## Step 1(b): Text Splitting

In [6]:
text_spliter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_spliter.create_documents([easy])
print(chunks)
print("\n")
print(len(chunks))
print("\n")
print(chunks[100].page_content)

[Document(metadata={}, page_content='यह एक अद्भुत रात थी। ऐसी रात जो सिर्फ जवानी में ही मुमकिन होती है। प्रिय पाठक आसमान सितारों से भरा हुआ था। इतना चमकदार उसे देखकर कोई यह सोचने पर मजबूर हो जाए कि क्या कोई चिड़चिड़ा या सनके इंसान ऐसे आसमान के नीचे रह सकता है? यह भी एक युवा सोच है। प्रिय पाठक बहुत युवा सोच। लेकिन ईश्वर करे कि यह सवाल आपके दिल में भी बार-बार उठे। जब मैं सनकी और चिड़चिड़े लोगों की बात करता हूं तो मैं अपने ही मन की हालत को याद करने से खुद को रोक नहीं सकता। पूरा दिन मेरा मन अजीब सी उदासी से भरा रहा। सुबह से ही ऐसा लगा मानो मैं अकेला हूं जैसे सब कुछ सब कोई मुझे छोड़कर चले जा रहे हो। अब कोई मुझसे पूछ सकता है कि सब कौन? मैं आठ साल से पीटर्सबर्ग में रह रहा था। लेकिन मेरी कोई खास जान पहचान नहीं थी। लेकिन मुझे लोगों की जान पहचान की जरूरत भी क्या थी? मैं पीटर्सबर्ग को वैसे ही जानता था जैसे कोई किसी दोस्त को जानता है। यही वजह है कि जब गर्मी के दिनों में पूरा शहर अपना सामान समेट कर गांव चला जाता तो मुझे लगता कि सब मुझे अकेला छोड़कर चले गए हैं। अकेले रह जाने का डर मुझे सताने लगा। ती

## Step 1(c): Embedding generation

In [7]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

## Step 1(d): Storing in  Vector Store

In [8]:
vector_store = FAISS.from_documents(chunks, embeddings)

In [10]:
vector_store.index_to_docstore_id

{0: '26499579-52fc-444e-ba17-0ee607f5c8f4',
 1: '6642a04a-e3e7-4a4e-8349-d9955ed257bb',
 2: 'a3876991-8447-4219-a9f4-45977c468ce6',
 3: '6d79cfde-904b-47ba-ab97-70299fc2bb6f',
 4: '24b53bf3-4e23-4556-a011-5f42c6e711db',
 5: '26b9374f-a73c-4c96-94bb-a398fea9612b',
 6: 'af91fdaa-df7d-4c35-9bb4-c7d8b8f786d2',
 7: 'e27a24ce-331c-4a84-b2cf-49b90c8d2aa4',
 8: '74d08453-dbff-4114-9355-ff030758c2e3',
 9: '6b91f50d-7185-4a9f-b324-125479e06412',
 10: 'f8ffb320-10c5-4981-9df2-53138aedfb90',
 11: '02a6a21e-d78a-48ed-9299-0b06332bfa53',
 12: '0110afa0-33a5-4e4e-85a0-547475464496',
 13: '8063a491-8e18-4a7e-9ef7-20e54d109733',
 14: 'cc43c524-8827-47fb-96e1-60bbfcdeabdf',
 15: '8c0460a9-5772-43a5-b627-4c80002e30e5',
 16: 'b844d280-0787-4c62-a34c-ea6add323b41',
 17: '26fd1201-6473-44b5-a5a1-a0b726346b8a',
 18: 'f8b2b260-c264-485a-b397-3fe91d5beb42',
 19: 'cadc35c4-c571-4302-bfcf-25532876f8a5',
 20: '36b3be96-ff9a-48eb-aaba-0d0e1ca03f0d',
 21: '5f2ef969-ffcb-423d-a478-76e01b314985',
 22: '0fff3470-36d5-

In [11]:
vector_store.get_by_ids(['818a2d2e-b0bd-4196-95ef-1eca9fc63eee'])

[]

## Step 2: Retrieval

In [12]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={'k':4}
)

In [13]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020F094BE540>, search_kwargs={'k': 4})

In [14]:
retriever.invoke('')

[Document(id='9e155a73-0c47-48de-a942-b489ecb25627', metadata={}, page_content='बेहतर जानेंगे। हां, और कल मैं भी तुम्हें अपने बारे में सब कुछ बता दूंगा। लेकिन यह सब क्या हो रहा है? जैसे कोई चमत्कार हुआ हो। हे भगवान, मैं कहां हूं? बताओ क्या तुम खुश नहीं हो कि तुमने मुझे पहले ही पल में ठुकराया नहीं। किसी और लड़की ने तो शायद ऐसा कर दिया होता। पर तुमने सिर्फ 2 मिनट में मुझे हमेशा के लिए खुश कर दिया। हां, सच में खुश। शायद तुमने मुझे खुद से सुलह करने का रास्ता दिखा दिया। शायद तुमने मेरे सारे संदेह मिटा दिए। शायद यह वह पल है जो मुझे चाहिए था। पर रहने दो। कल मैं तुम्हें सब बता दूंगा। तुम सब कुछ जान जाओगे सब कुछ। ठीक है। मैं राजी हूं। लड़की ने धीरे से कहा, कल तुम शुरू करोगे। तैरा रहा तो फिर कल मिलते हैं। कल मिलते हैं। हम अलग हो गए। मैं पूरी रात यूं ही सड़कों पर घूमता रहा। मेरा दिल घर जाने के लिए तैयार ही नहीं था। मैं बहुत खुश था। कल तो तुम जिंदा हो। उसने मेरे दोनों हाथ दबाते हुए कहा, मैं यहां पिछले दो घंटे से हूं। तुम नहीं जानते मैं पूरा दिन कितनी बेचैनी में रही। मुझे पता है, मुझे पता है। लेकिन

## Step 3: Augmentation

In [15]:
llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0.2
)

In [16]:
prompt = PromptTemplate(
    template="""
        आप उपन्यास **"White Nights"** के लेखक की भूमिका निभा रहे हैं।
        नीचे दिए गए संदर्भ के आधार पर ही उत्तर दें।
        

        संदर्भ:
        {context}

        प्रश्न: {question}

    """,
    input_variables= ['context', 'question']
)

In [17]:
question = "कहानी की चार रातों में क्या-क्या मुख्य घटनाएँ होती हैं?"
retrieved_docs = retriever.invoke(question)

In [18]:
retrieved_docs

[Document(id='03a5d637-44eb-4826-985f-cfbae78491e1', metadata={}, page_content='चाहता कि तुम्हारी बाकी तकलीफों के अलावा मैं कोई और दुख बनूं। यह मेरी ही गलती है नास्तिन का। अच्छा अलविदा। रुको मेरी बात सुनो। क्या तुम इंतजार कर सकते हो? किसका? कैसे? मैं उससे प्यार करती हूं लेकिन मैं इस दर्द से उभर जाऊंगी। मुझे उभरना ही होगा। मैं इसे भूल जाऊंगी। मुझे यह करना ही होगा। मैं पहले से ही इसे महसूस कर रही हूं। कौन जानता है? शायद यह सब आज ही खत्म हो जाए। क्योंकि उसने मुझ पर हंसा जबकि तुम मेरे साथ रोए क्योंकि तुमने मुझे नहीं ठुकराया जैसा कि उसने किया क्योंकि तुम मुझसे प्यार करते हो जबकि उसने कभी मुझसे प्यार नहीं किया क्योंकि मैं भी तुमसे प्यार करती हूं। हां, मैं तुमसे प्यार करती हूं। मैं तुमसे वैसे ही प्यार करती हूं जैसे तुम मुझसे करते हो। मैंने पहले भी कहा था तुमने खुद सुना था मैं तुमसे प्यार करती हूं। क्योंकि तुम मुझसे बेहतर हो क्योंकि तुम मुझसे बड़े हो क्योंकि क्योंकि वह और वह कुछ नहीं कह सकी। उसके आंसू फिर से बह निकले। उसने अपना सिर मेरे कंधे पर रखा। फिर मेरे सीने पर और फूट-फूट कर रोने लगी। मैं

In [19]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'चाहता कि तुम्हारी बाकी तकलीफों के अलावा मैं कोई और दुख बनूं। यह मेरी ही गलती है नास्तिन का। अच्छा अलविदा। रुको मेरी बात सुनो। क्या तुम इंतजार कर सकते हो? किसका? कैसे? मैं उससे प्यार करती हूं लेकिन मैं इस दर्द से उभर जाऊंगी। मुझे उभरना ही होगा। मैं इसे भूल जाऊंगी। मुझे यह करना ही होगा। मैं पहले से ही इसे महसूस कर रही हूं। कौन जानता है? शायद यह सब आज ही खत्म हो जाए। क्योंकि उसने मुझ पर हंसा जबकि तुम मेरे साथ रोए क्योंकि तुमने मुझे नहीं ठुकराया जैसा कि उसने किया क्योंकि तुम मुझसे प्यार करते हो जबकि उसने कभी मुझसे प्यार नहीं किया क्योंकि मैं भी तुमसे प्यार करती हूं। हां, मैं तुमसे प्यार करती हूं। मैं तुमसे वैसे ही प्यार करती हूं जैसे तुम मुझसे करते हो। मैंने पहले भी कहा था तुमने खुद सुना था मैं तुमसे प्यार करती हूं। क्योंकि तुम मुझसे बेहतर हो क्योंकि तुम मुझसे बड़े हो क्योंकि क्योंकि वह और वह कुछ नहीं कह सकी। उसके आंसू फिर से बह निकले। उसने अपना सिर मेरे कंधे पर रखा। फिर मेरे सीने पर और फूट-फूट कर रोने लगी। मैंने उसे चुप कराने की कोशिश की। उसे दिलासा दिया। लेकिन वह रोना बंद नहीं कर पाई।\n

In [20]:
final_prompt = prompt.invoke({'context': context_text, "question": question})

In [21]:
final_prompt

StringPromptValue(text='\n        आप उपन्यास **"White Nights"** के लेखक की भूमिका निभा रहे हैं।\n        नीचे दिए गए संदर्भ के आधार पर ही उत्तर दें।\n\n\n        संदर्भ:\n        चाहता कि तुम्हारी बाकी तकलीफों के अलावा मैं कोई और दुख बनूं। यह मेरी ही गलती है नास्तिन का। अच्छा अलविदा। रुको मेरी बात सुनो। क्या तुम इंतजार कर सकते हो? किसका? कैसे? मैं उससे प्यार करती हूं लेकिन मैं इस दर्द से उभर जाऊंगी। मुझे उभरना ही होगा। मैं इसे भूल जाऊंगी। मुझे यह करना ही होगा। मैं पहले से ही इसे महसूस कर रही हूं। कौन जानता है? शायद यह सब आज ही खत्म हो जाए। क्योंकि उसने मुझ पर हंसा जबकि तुम मेरे साथ रोए क्योंकि तुमने मुझे नहीं ठुकराया जैसा कि उसने किया क्योंकि तुम मुझसे प्यार करते हो जबकि उसने कभी मुझसे प्यार नहीं किया क्योंकि मैं भी तुमसे प्यार करती हूं। हां, मैं तुमसे प्यार करती हूं। मैं तुमसे वैसे ही प्यार करती हूं जैसे तुम मुझसे करते हो। मैंने पहले भी कहा था तुमने खुद सुना था मैं तुमसे प्यार करती हूं। क्योंकि तुम मुझसे बेहतर हो क्योंकि तुम मुझसे बड़े हो क्योंकि क्योंकि वह और वह कुछ नहीं कह सकी। उसके

## Step 4: Generation

In [22]:
response = llm.invoke(final_prompt)
print(response.content)

चार रातों में नास्तिनका और उसके दोस्त के बीच एक गहरा प्रेम विकसित हो जाता है। नास्तिनका अपने दिल की बातें साझा करती है और अपने दोस्त को बताती है कि वह उसे बहुत पसंद करती है, लेकिन उसके पत्र मिलने के बाद भी वे एक-दूसरे के प्रति अपने प्यार को साझा नहीं कर पाते।


# Building a Chain